In [7]:
!pip install -q transformers datasets peft accelerate evaluate scikit-learn pandas sentencepiece

In [8]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
from peft import get_peft_model, LoraConfig, TaskType
import evaluate
import os

# Check for MPS (Apple Silicon GPU) or CUDA
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")


Using CUDA GPU


In [9]:
# Load Data
train_df = pd.read_csv('/kaggle/input/competitions/dlp-nppe-1-t-22026/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/dlp-nppe-1-t-22026/test.csv')
sample_sub = pd.read_csv('/kaggle/input/competitions/dlp-nppe-1-t-22026/sample_submission.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# 80/20 Train/Validation Split
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['label'])
print(f"Train set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)


Train shape: (50840, 3)
Test shape: (12710, 2)
Train set shape: (40672, 3)
Validation set shape: (10168, 3)


In [10]:
from transformers import AutoTokenizer
MODEL_NAME = "microsoft/deberta-v3-base" 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


Map:   0%|          | 0/40672 [00:00<?, ? examples/s]

Map:   0%|          | 0/10168 [00:00<?, ? examples/s]

Map:   0%|          | 0/12710 [00:00<?, ? examples/s]

In [12]:
!pip uninstall torchao --y

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [13]:
num_labels = train_df['label'].nunique()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=num_labels
)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=16, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules="all-linear" 
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

trainable params: 2,701,854 || all params: 187,147,068 || trainable%: 1.4437


In [14]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-4, 
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    use_cpu=False, 
    report_to="none" 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,169.841438,76.220253,0.098348
2,101.221859,68.714355,0.083989
3,68.185914,38.595619,0.260917
4,60.723301,28.425777,0.065106
5,48.603070,21.769999,0.138867


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=6355, training_loss=101.30605806820417, metrics={'train_runtime': 8317.6726, 'train_samples_per_second': 24.449, 'train_steps_per_second': 0.764, 'total_flos': 5.520858272169984e+16, 'train_loss': 101.30605806820417, 'epoch': 5.0})

In [15]:

predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)

submission = pd.DataFrame({
    'ID': test_df['id'],
    'label': preds
})

submission.to_csv('submission.csv', index=False)
print("Saved submission to submission.csv")
submission.head()


Saved submission to submission.csv


,ID,label
0,0,7
1,1,7
2,2,7
3,3,7
4,4,7
